# Training 3 different RNNs

In this notebook I will train 3 different RNNs, one simple RNN, one LSTM and one GRU.

In [1]:
import pandas as pd
import numpy as np
import os
import augumentations as aug
import data_functions as dfunc
import torch

folder_path = "../../MainProject/data/mediapipe_not_trimmed_world"
score_path = "../../MainProject/data/video_scores.csv"

## Data Creation

We begin by creating the data we need as well as augumenting it by mirroring and rotating the node network.

In [2]:
scores = dfunc.load_video_score(score_path=score_path)
files = list(scores["file"])
labels = torch.tensor(list(scores["scaled_score"]))

# First and last rotate nothing
rotations = np.linspace(0, 2*np.pi, 5)[:-1]

# Extend the labels to add target cariables for the augumented data
factor = 2 * len(rotations)
labels = dfunc.extend_tensor(labels, factor)

samples = []
for index, file in enumerate(files):
    path = os.path.join(folder_path, f"{file}_mediapipe.csv")

    # Select frames from the file
    df = pd.read_csv(path).drop(columns=["FrameNo"])
    df_sliced = dfunc.select_equally_spaced_rows(df)

    # Augument the data
    for mirroring in [True, False]:
        mirrored_data = aug.mirror(df_sliced, mirror_x=mirroring)

        for angle in rotations:
            rotated_data = aug.rotate(mirrored_data, angle, axis=0)

            # Convert to tensors
            sample = dfunc.create_tensor_from_dataframe(rotated_data)
            samples.append(sample)

print(len(samples))
samples = torch.stack(samples)
print(samples.shape)
data = dfunc.create_TensorDataset(samples, labels)
if data is not None:
    print("Successfully created DataLoader!")
else:
    print("Something went wrong!")

1400
torch.Size([1400, 30, 39])
Successfully created DataLoader!
